# PaliGemma 2: bul, kes, tekrar oku (30 fiş)

Bu deneyde PaliGemma 2 3B'nin mix-224 ve mix-448 modelleri fişlerdeki toplam tutarı okudu. Önce 5 fişle küçük bir deneme yapıldı. Sonuçları daha güvenilir kılmak için deney 30 fişe genişletildi. Bu notebook 30 fişlik deneyi içerir.

| Yöntem | Ne yapıyor | Gemma'ya giden resim parçası sayısı |
|---|---|---|
| **A** | mix-224, bütün fiş | 256 |
| **B** | mix-448, bütün fiş | 1.024 |
| **C** | mix-224: `detect total` / `detect receipt` ile bölgeyi bul → orijinal fotoğraftan kes → tekrar 224'te oku | 256 + 256 = 512 |
| **C\*** | Kontrol: toplam fiyatın yazdığı yer elle verilir, kesilip 224'te okunur (bulma adımı devre dışı). Üç kesit şekli denendi; ana şekil olarak "bağlamlı" deneyden önce seçildi. | 256 |

Modele sorulan soru: "What is the total amount?" Fiş fotoğrafları Wikimedia Commons'tan alındı (kamu malı, CC0 ve CC BY 4.0) ve aynı klasördeki `fisler30.zip` dosyasında. Fotoğrafçılar ve lisanslar `KAYNAKLAR.md` dosyasında.

In [ ]:
# 0) Hugging Face girişi. Token kodda yazmaz: HF_TOKEN ortam değişkeninden ya da `hf auth login` kaydından okunur.
#    Colab tarayıcıda çalıştırılıyorsa: from google.colab import userdata; import os; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import get_token
print("Hugging Face girişi:", "var" if get_token() else "YOK")

Hugging Face girişi: var


In [ ]:
# 1) Kontrol ve yardımcı fonksiyonlar
!nvidia-smi --query-gpu=name,memory.total --format=csv
import re, os, gc, io, time, json, urllib.request, urllib.parse, torch
import pandas as pd
from PIL import Image, ImageDraw
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor
from huggingface_hub import model_info
for mid in ["google/paligemma2-3b-mix-224", "google/paligemma2-3b-mix-448"]:
    try: model_info(mid); print("erişim var:", mid)
    except Exception as e: print("ERİŞİM YOK:", mid, type(e).__name__)

def load(model_id):
    proc = PaliGemmaProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, dtype=torch.bfloat16, device_map="cuda").eval()
    return model, proc

def ask(model, proc, image, prompt, max_new_tokens=32):
    inputs = proc(text="<image>" + prompt, images=image.convert("RGB"), return_tensors="pt").to(model.device)
    inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    L = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][L:]
    return proc.decode(gen, skip_special_tokens=True).strip(), proc.decode(gen, skip_special_tokens=False)

def parse_box(raw, W, H):
    v = [int(x) for x in re.findall(r"<loc(\d{4})>", raw)]
    if len(v) < 4: return None
    y0, x0, y1, x1 = v[:4]
    return (x0 / 1024 * W, y0 / 1024 * H, x1 / 1024 * W, y1 / 1024 * H)

def square_crop(img, box, pad=0.05):
    W, H = img.size; x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(max(x1 - x0, y1 - y0) * (1 + 2 * pad), 0.08 * min(W, H))
    l, t, s = int(cx - side / 2), int(cy - side / 2), int(side)
    canvas = Image.new("RGB", (s, s), "white")
    canvas.paste(img.crop((max(0, l), max(0, t), min(W, l + s), min(H, t + s))), (max(0, -l), max(0, -t)))
    return canvas

def stretch_crop(img, box):
    return img.crop(tuple(int(v) for v in box))

def context_crop(img, box):
    """Ana kesit şekli: kutu kısa kenarı boyunca uzun kenarın yarısına genişletilir (üst/alt satırlar da girer)."""
    W, H = img.size; x0, y0, x1, y1 = box; w, h = x1 - x0, y1 - y0
    if w > h:
        cy = (y0 + y1) / 2; nh = max(h, w * 0.5); y0, y1 = max(0, cy - nh / 2), min(H, cy + nh / 2)
    else:
        cx = (x0 + x1) / 2; nw = max(w, h * 0.5); x0, x1 = max(0, cx - nw / 2), min(W, cx + nw / 2)
    return img.crop((int(x0), int(y0), int(x1), int(y1)))

def to_number(s, dot_thousands=False):
    s = s.replace(" ", "")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif "," in s:
        s = s.replace(",", ".") if len(s.split(",")[-1]) in (1, 2) else s.replace(",", "")
    elif s.count(".") > 1 or (dot_thousands and s.count(".") == 1 and len(s.split(".")[-1]) == 3):
        s = s.replace(".", "")
    return float(s)

def is_correct(answer, truth):
    """Cevaptaki herhangi bir sayı doğru toplama eşitse doğru. '13.000' gibi nokta-binlik yazımlar da kabul edilir."""
    t = to_number(truth)
    for m in re.findall(r"\d[\d.,]*\d|\d", answer):
        for dt in (False, True):
            try:
                if abs(to_number(m, dt) - t) < 0.011: return True
            except ValueError: pass
    return False

assert is_correct("76,000", "76,000") and is_correct("13.000", "13,000") and is_correct("$ 26.54", "26.54") and not is_correct("26.19", "26.54")
print("hazır")

name, memory.total [MiB]
Tesla T4, 15360 MiB


erişim var: google/paligemma2-3b-mix-224
erişim var: google/paligemma2-3b-mix-448
hazır


In [ ]:
# 2) 30 fiş: Wikimedia Commons, kamu malı / CC0 / CC BY 4.0. Doğru toplamlar ve toplamın yazdığı yer (kutu) elle belirlendi.
#    Görüntüler Wikimedia'dan indirilip en uzun kenarı en fazla 3000 piksele küçültüldü ve fisler30.zip olarak yüklendi
#    (Colab'dan doğrudan indirme Wikimedia hız sınırına takılıyor). Kutular bu görüntülere göredir.
import zipfile
RECEIPTS = [
  ('r01', "File:Alexander's Supermarket receipt, late twentieth century.tif", '39.99', (120, 1245, 640, 1305), 'CC0 · Struthious Bandersnatch'),
  ('r02', 'File:Grocery Store Receipt in Vienna.jpg', '22,41', (1700, 650, 1900, 1560), 'Public domain · Grandmaster Huon'),
  ('r03', 'File:Tesco grocery receipt Finchley 1994.jpg', '6.71', (80, 1320, 710, 1390), 'Public domain · Tesco'),
  ('r04', 'File:Kassenzettel Fressnapf mit ausgewiesener MwSt-Senkung und TSE Transaktionsnummer 12 2020.png', '34,09', (0, 1055, 1078, 1140), 'Public domain · Fressnapf'),
  ('r05', 'File:Receipt from the store (PNG).jpg', '19.50', (330, 870, 1180, 940), 'CC BY 4.0 · Виктор Пинчук'),
  ('r06', 'File:Receipt taebaek restaurant IMG 2614 modified.jpg', '76,000', (0, 1845, 2192, 1945), 'Public domain · 이태리집(태백시 소재)'),
  ('r07', 'File:Cosmed Baifu Store paying receipt 20170117.jpg', '110', (95, 295, 480, 352), 'Public domain · Solomon203'),
  ('r08', 'File:MaxValu Matsuyama Store receipt JPY254 20191109.jpg', '246', (140, 835, 600, 882), 'Public domain · Solomon203'),
  ('r09', 'File:Taiwan Uniqlo Miramar Store receipt 2012-05-07.jpg', '599', (100, 615, 520, 680), 'Public domain · Solomon203'),
  ('r10', 'File:Receipt from Lawson in South Tangerang, Indonesia with QRIS CPM (2025-09-02).jpg', '13,000', (190, 700, 1120, 765), 'CC BY 4.0 · VulcanSphere'),
  ('r11', 'File:A receipt in Thai from a restaurant in Chiang Mai (2019).jpg', '1,362.00', (550, 2690, 1700, 2900), 'CC BY 4.0 · Chainwit.'),
  ('r12', 'File:HK SYP 西環 Sai Ying Pun 德輔道西 308 Des Voeux Road West 食神麗宮酒家 Chinese Banquet Seafood Restaurant bill receipt January 2026 N13P 02.jpg', '985.0', (650, 1430, 1760, 1640), 'CC0 · EVOW Sahgyune Paiem KAASEI 02238'),
  ('r13', 'File:JP 日本 Japan 京都市Kyoto 中华料理店 餃子の王将 Gyoza no Ohsho Japanese Chinese restaurant chain Receipt in June 2026 N13P 01.jpg', '2,033', (230, 720, 620, 795), 'CC0 · TKdows 2026'),
  ('r14', 'File:Arco 3076 receipt (2003-09-16).jpg', '26.54', (80, 1220, 600, 1290), 'Public domain · Amin Eshaiker'),
  ('r15', 'File:Arco 9714 receipt (2003-10-20).jpg', '24.01', (70, 1285, 600, 1355), 'Public domain · Amin Eshaiker'),
  ('r16', 'File:Circle K 5705 receipt (2005-07-22).jpg', '25.99', (40, 1080, 699, 1150), 'Public domain · Amin Eshaiker'),
  ('r17', 'File:NB Oil Co (Shell) 1 receipt (2007-02-18).jpg', '20.87', (100, 1800, 620, 1865), 'Public domain · Amin Eshaiker'),
  ('r18', 'File:Shell-Gas-Station-Receipt-MasterCard.jpg', '20.00', (0, 990, 915, 1120), 'Public domain · Mrmiscellanious~enwiki'),
  ('r19', 'File:Bill Vishwanand 2026-01-28.jpg', '525', (950, 2200, 1780, 2400), 'CC0 · Sakurakat'),
  ('r20', 'File:Fanzz, Westminster Mall receipt (2002-12-26).jpg', '8.61', (370, 1190, 880, 1260), 'Public domain · Amin Eshaiker'),
  ('r21', 'File:Shiekh Shoes at Westminster Mall receipt (2002-06-19).jpg', '86.17', (500, 828, 800, 885), 'Public domain · Amin Eshaiker'),
  ('r22', 'File:USPS Receipt - 2002-01-31.jpg', '1.55', (25, 468, 440, 505), 'Public domain · Amin Eshaiker'),
  ('r23', 'File:Paragon fiskalny-30.10.2025.jpg', '62,00', (30, 1093, 880, 1160), 'CC BY 4.0 · Aw58'),
  ('r24', 'File:Paragon fiskalny- 14.05.2025.jpg', '58,63', (20, 948, 960, 1020), 'CC BY 4.0 · Aw58'),
  ('r25', 'File:Paragon fiskalny -24.06.2021.jpg', '645.50', (0, 975, 668, 1050), 'CC BY 4.0 · Aw58'),
  ('r26', 'File:Paragon fiskalny - 17.02.2021.jpg', '299.99', (10, 550, 926, 615), 'CC BY 4.0 · Aw58'),
  ('r27', 'File:Paragon fiskalny - 17.02.2025.jpg', '61,90', (5, 715, 463, 770), 'CC BY 4.0 · Aw58'),
  ('r28', 'File:Paragon fiskalny - 21.02.2026.jpg', '127,98', (60, 790, 930, 860), 'CC BY 4.0 · Aw58'),
  ('r29', 'File:Paragon fiskalny - 25.03.2026.jpg', '29,99', (20, 685, 951, 755), 'CC BY 4.0 · Aw58'),
  ('r30', 'File:Paragon fiskalny - CCI24072025.jpg', '143,98', (15, 680, 693, 740), 'CC BY 4.0 · Aw58'),
]
EXPECTED = {'r01': (768, 1552), 'r02': (3000, 2250), 'r03': (769, 2060), 'r04': (1078, 3000), 'r05': (1506, 2007), 'r06': (2192, 3000),
            'r07': (600, 600), 'r08': (720, 1280), 'r09': (629, 1600), 'r10': (1274, 1522), 'r11': (2250, 3000), 'r12': (2250, 3000),
            'r13': (750, 1000), 'r14': (676, 1812), 'r15': (669, 1860), 'r16': (699, 1659), 'r17': (672, 2192), 'r18': (915, 1415),
            'r19': (2258, 3000), 'r20': (947, 1734), 'r21': (968, 1252), 'r22': (492, 818), 'r23': (960, 1431), 'r24': (972, 1695),
            'r25': (668, 1436), 'r26': (926, 1214), 'r27': (463, 1024), 'r28': (969, 1434), 'r29': (951, 1409), 'r30': (693, 1417)}
os.makedirs("/content/receipts", exist_ok=True)
zipfile.ZipFile("/content/fisler30.zip").extractall("/content/receipts")
DATA = {}
for key, title, total, box, lic in RECEIPTS:
    fn = f"/content/receipts/{key}.jpg"; size = Image.open(fn).size
    assert size == EXPECTED[key], (key, size)          # her dosyanın doğru fiş olduğundan emin ol
    DATA[key] = dict(path=fn, total=total, box=box, lic=lic)
print(len(DATA), "fiş yüklendi, boyutlar beklenenle aynı.")

30 fiş yüklendi, boyutlar beklenenle aynı.


In [ ]:
# 3) mix-224: A (bütün fiş), C (detect ile bul + kes), C* (bölge elle verilir; ana kesit şekli: bağlamlı)
Q = "answer en What is the total amount?"
rows = []
m224, p224 = load("google/paligemma2-3b-mix-224")
for key, d in DATA.items():
    img = Image.open(d["path"]).convert("RGB"); W, H = img.size
    add = lambda method, ans, **kw: rows.append(dict(fis=key, yontem=method, cevap=ans, dogru=is_correct(ans, d["total"]), **kw))
    a, _ = ask(m224, p224, img, Q); add("A: 224 bütün fiş", a, parca=256)
    for target in ["total", "receipt"]:
        _, raw = ask(m224, p224, img, f"detect {target}")
        box = parse_box(raw, W, H)
        if box is None:
            rows.append(dict(fis=key, yontem=f"C: detect {target}", cevap="(kutu bulunamadı)", dogru=False, parca=512)); continue
        whole = (box[2] - box[0]) * (box[3] - box[1]) > 0.8 * W * H
        c, _ = ask(m224, p224, context_crop(img, box), Q)
        add(f"C: detect {target}", c + ("  [kutu ≈ bütün resim]" if whole else ""), parca=512)
    for name, fn in [("C*: bağlamlı", context_crop), ("C*: gerdir", stretch_crop), ("C*: kare+boşluk", square_crop)]:
        c, _ = ask(m224, p224, fn(img, d["box"]), Q); add(name, c, parca=256)
    print(key, "tamam", flush=True)
del m224, p224; gc.collect(); torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

r26 tamam


r27 tamam


r28 tamam


r29 tamam


r30 tamam


In [ ]:
# 4) mix-448: B (bütün fiş)
m448, p448 = load("google/paligemma2-3b-mix-448")
for key, d in DATA.items():
    b, _ = ask(m448, p448, Image.open(d["path"]).convert("RGB"), Q)
    rows.append(dict(fis=key, yontem="B: 448 bütün fiş", cevap=b, dogru=is_correct(b, d["total"]), parca=1024))
del m448, p448; gc.collect(); torch.cuda.empty_cache()
print("tamam")

preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tamam


In [ ]:
# 5) Sonuçlar
df = pd.DataFrame(rows)
order = ["A: 224 bütün fiş", "B: 448 bütün fiş", "C: detect total", "C: detect receipt", "C*: bağlamlı", "C*: gerdir", "C*: kare+boşluk"]
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 30)
tab = df.pivot(index="fis", columns="yontem", values="cevap")[order]
tab.insert(0, "dogru_toplam", [DATA[k]["total"] for k in tab.index])
print(tab.to_string()); print()
summary = df.groupby("yontem").agg(dogru=("dogru", "sum"), parca=("parca", "first")).reindex(order)
summary["oran"] = (summary.dogru / len(DATA) * 100).round(0).astype(int).astype(str) + "%"
print(summary.to_string())
os.makedirs("/content/out30", exist_ok=True)
df.to_csv("/content/out30/sonuclar_30fis.csv", index=False)

yontem dogru_toplam A: 224 bütün fiş B: 448 bütün fiş                 C: detect total  C: detect receipt C*: bağlamlı C*: gerdir C*: kare+boşluk
fis                                                                                                                                             
r01           39.99            37.85            39.85               (kutu bulunamadı)  (kutu bulunamadı)        39.85      33.93           48.88
r02           22,41            18.00            22.41               (kutu bulunamadı)  (kutu bulunamadı)        22.41      22.41           22.41
r03            6.71             4.71             6.71      4.71  [kutu ≈ bütün resim]  (kutu bulunamadı)         6.71       4.71            6.71
r04           34,09            97.87            34.97    134.97  [kutu ≈ bütün resim]  (kutu bulunamadı)        20.06      34.03           75,08
r05           19.50            20.50            19.50               (kutu bulunamadı)  (kutu bulunamadı)        19.50       4.35  